# 2.1.11 — Numerical Computing Principles and Limitations

**Module:** 02 — NumPy: Advanced Numerical Computing and Array Programming  
**Chapter:** 2.1 — NumPy Architecture and Fundamentals  
**Level:** Expert / Research-Oriented Foundations  
**Prerequisites:** Advanced Python object semantics; basic linear algebra; numerical analysis fundamentals.

> **Purpose.** Develop a precise, testable conceptual and computational model of numerical computing principles and limitations. This notebook separates mathematical intent, Python-level semantics, storage consequences, and execution behavior. It is a standalone section of the Chapter 2.1 architecture sequence.

## 1. Learning Objectives

After studying this notebook, you should be able to:

1. State the central abstractions and identify the level of the system at which each guarantee applies.
2. Explain the relevant mathematical or computational invariants without relying on informal analogies.
3. Construct executable tests that distinguish correct behavior from plausible but invalid assumptions.
4. Identify boundary cases, side effects, dtype concerns, and complexity implications.
5. Translate the topic into a reproducible, documented numerical research workflow.

## 2. Reproducibility and Environment

The examples use the standard `numpy` package and Python standard library. They are intended for recent NumPy 2.x installations; most examples also work on current 1.26 releases, but NumPy 2.x promotion and interface details should be interpreted with the installed version in mind. Execute cells **top to bottom**. There are no external datasets, downloads, or network dependencies.

In [ ]:
import numpy as np
import sys

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
assert tuple(map(int, np.__version__.split('.')[:2])) >= (1, 24), 'Use a recent NumPy release.'

## 3. Conceptual Foundations and Formal Semantics

### 1. Finite representation

Binary floating-point numbers approximate real numbers using sign, exponent, and significand. Most decimal fractions do not have exact finite binary representations. NumPy vectorization changes the execution strategy but not these fundamental limitations.

### 2. Error metrics

For exact `x` and computed `x_hat`, absolute error is `|x_hat-x|`; relative error is meaningful when `x != 0`. Forward error measures the deviation in output; backward-error analysis asks whether the computed answer solves a nearby problem. Conditioning is a property of the problem, not merely the algorithm.

### 3. Nonassociativity

Floating-point addition is not associative in general: `(a+b)+c` can differ from `a+(b+c)`. Consequently, reductions performed in different orders or by different libraries may not match bit-for-bit. Reproducible numerical research should use explicitly justified error tolerances.

### 4. Stability and cancellation

Subtracting nearby large floating-point values can lose significant digits; unstable formulas can amplify small errors. An algorithm can be mathematically exact on paper but operationally inaccurate in finite precision. DType upgrades can help but are not a substitute for stable algebraic formulations.

### 5. Domain-specific limitations

NumPy's core does not enforce arbitrage constraints, exchange calendars, missing-data conventions, market microstructure, or model identification. Financial models must validate these assumptions outside the raw array machinery. Numeric accuracy and economic validity are separate concerns.

## 4. Architectural Model and Invariants

A useful four-layer checklist for this topic:

| Layer | Question to ask | Representative invariant |
|---|---|---|
| Logical | What operation is mathematically intended? | Axes and result shape correspond to the model |
| Representation | What dtype and storage model represent the inputs? | No unsupported casting or unexpected aliasing |
| Execution | Which category of operation processes the arrays? | Results are stable under a reference implementation |
| Validation | What independent evidence establishes correctness? | Assertions, boundary tests, and numerical tolerances |

**Important:** Shape agreement and code execution are not sufficient evidence of mathematical validity. Conversely, a mathematically valid expression can be operationally impractical or numerically unstable.

## 5. Executable Investigations

Each investigation is independent in purpose; execute in sequence so shared imports are available.

### 5.1 — Nonassociativity as an observable property

**Research question.** What does this experiment establish, and which conclusion would be unjustified if we only inspected its printed values?

In [ ]:
import numpy as np
a, b, c = np.float64(1e16), np.float64(-1e16), np.float64(1.)
lhs, rhs = (a+b)+c, a+(b+c)
print('(a+b)+c:', lhs, 'a+(b+c):', rhs)
assert lhs != rhs

### 5.2 — Loss of precision in small increments

**Research question.** What does this experiment establish, and which conclusion would be unjustified if we only inspected its printed values?

In [ ]:
large = np.float64(1e16)
small = np.float64(1.)
print('large + small == large:', large + small == large)
assert large + small == large

### 5.3 — Stable norm and error validation

**Research question.** What does this experiment establish, and which conclusion would be unjustified if we only inspected its printed values?

In [ ]:
v = np.array([1e200, 1e200])
naive = np.sqrt(np.sum(v*v))
stable = np.hypot(v[0], v[1])
print('naive:', naive, 'scaled hypot:', stable)
assert np.isinf(naive) and np.isfinite(stable)
np.testing.assert_allclose(stable / 1e200, np.sqrt(2), rtol=1e-15)

## 6. Edge Cases, Failure Modes, and Misconceptions

- **Logical validity is not physical contiguity.** Shape alone cannot determine whether an operation needs a copy.
- **Value equality is not identity.** Two arrays may compare equal while having independent storage, or share storage while being distinct Python objects.
- **A vectorized expression does not certify correctness.** Broadcasting and axis conventions can silently encode unintended mathematics.
- **A dtype is not a real-number model.** Overflow, finite precision, and representation-dependent transformations require explicit attention.
- **A benchmark does not imply a theorem.** Hardware, build configuration, allocation strategy, and input scale can affect conclusions.
- **Array semantics do not encode domain assumptions.** For finance or statistics, look-ahead restrictions, missing-data rules, weights, and alignment remain external responsibilities.

When evaluating an unexpected result, first inspect `shape`, `dtype`, `strides`, and storage overlap, then isolate the mathematical operation on a hand-checkable example.

## 7. Performance and Complexity Considerations

The performance question should be asked in three parts: **work complexity**, **data movement**, and **peak live memory**. A contiguous numeric array can support efficient native processing, but a particular operation may be limited by memory bandwidth, temporary allocation, scalar dispatch, or native-library parallelism. Use measurements only after verifying correctness, and never assume that a microbenchmark generalizes to different dtypes or problem sizes.

A practical benchmark should (i) isolate the operation, (ii) distinguish input setup from computation, (iii) warm up the relevant execution path, (iv) collect repeated measurements, and (v) record platform and NumPy provenance.

## 8. Analytical Interpretation

A reliable numerical workflow distinguishes (1) model error, (2) data error, (3) discretization error, and (4) floating-point error. Changing float64 to float32 might influence only one of these, and sometimes can materially alter model conclusions. Absolute accuracy requirements should follow the actual application.

## 9. Advanced Exercises

These are open-ended expert exercises. Provide an analytical justification, a minimal counterexample where useful, and an executable validation test.

**Exercise 1.** Distinguish condition number, forward error, and backward error for linear systems.

**Exercise 2.** Give an example where an apparently faster float32 pipeline violates a required risk metric tolerance.

**Exercise 3.** Propose a scientifically justified tolerance strategy for comparing portfolio covariance matrices.

## 10. Expected Results and Validation Strategy

1. Execute every example in order; all assertions should pass on an appropriately configured NumPy installation.
2. Compare printed shapes, types, and DTypes with the surrounding mathematical and architectural claims.
3. Where a local benchmark is included, record measured values but **do not** expect a universal numeric timing.
4. Modify input shapes, dtypes, and views to construct at least one failing case; document why the original contract no longer holds.
5. Implement an independent reference for one operation and compare using an error metric appropriate to its numerical domain.

**Acceptance criteria:** the student can explain the behavior without relying on a copied output, design a counterexample to an invalid inference, and describe relevant representation and performance limitations.

## Further Reading and Verification Sources

The following are authoritative public references. API details may vary across NumPy versions; verify against the installed version when reproducing an experiment.

- [NumPy: The N-dimensional array](https://numpy.org/doc/stable/reference/arrays.ndarray.html)
- [NumPy: ndarray reference](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.html)
- [NumPy: Data types](https://numpy.org/doc/stable/reference/arrays.dtypes.html)
- [NumPy: Broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html)
- [NumPy: Copies and views](https://numpy.org/doc/stable/user/basics.copies.html)
- [NumPy: Universal functions](https://numpy.org/doc/stable/reference/ufuncs.html)
- [NumPy: Thread safety](https://numpy.org/doc/stable/reference/thread_safety.html)
- [NumPy: C API](https://numpy.org/doc/stable/reference/c-api/index.html)

**Scope note:** This is Chapter 2.1 (architectural foundations). Detailed DType implementation, advanced indexing, stride tricks, microarchitecture, and native C extension development are treated in Chapters 2.3, 2.5, 2.7, 2.21 and 2.23 respectively.